In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd
from show import *

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/cross/hmi.M_45s.20240319_152445_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.M_45s.20241120_001630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20240319T152009_V202602220014_0443190639.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-stokes_20240319T152009_V202602220014_0443190639.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-vlos_20240319T152009_V202602220014_0443190639.fits.gz']

In [3]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xu, yu = s['xu'], s['yu']

In [10]:
file_hmi = files[0]
file_fdt = files[2]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

data_hmi = rebin(data_hmi, 6, update_header=header_hmi)

In [25]:
did = int(file_fdt.split('_')[-1].split('.')[0])
view_fdt = View.from_header(header_fdt).update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0], crota=header_fdt['CROTA'] + 0.25)
view_hmi = View.from_header(header_hmi)

transform = view_fdt.to_carrington() - view_hmi.to_carrington()
grid = crop_grid(xu, yu, header_fdt)
grid, _ = transform(grid)

xi, yi = grid
xi_, yi_ = np.floor(xi).astype(int), np.floor(yi).astype(int)




In [35]:
plt.figure(figsize=(10,10))
plt.imshow(xi_ - xi, 'gray')
plt.tight_layout()

In [5]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt, 'hmimag', vmin=-1000, vmax=1000)
plt.axis('off')
plt.tight_layout()

In [6]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi, 'hmimag', vmin=-1000, vmax=1000)
plt.axis('off')
plt.tight_layout()